# Task 4: Transformer Encoder Block

Complete encoder with multi-head attention, feed-forward, and residuals.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)

In [ ]:
class TransformerEncoderBlock(nn.Module):
    """Single Transformer Encoder Block."""
    
    def __init__(self, embed_dim, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        
        # Multi-head attention
        self.self_attention = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.attn_dropout = nn.Dropout(dropout)
        
        # Feed-forward network
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, embed_dim),
            nn.Dropout(dropout)
        )
        
        # Layer normalization
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        
    def forward(self, x, mask=None):
        # Self-attention with residual
        attn_output, _ = self.self_attention(x, x, x, attn_mask=mask)
        x = self.norm1(x + self.attn_dropout(attn_output))
        
        # Feed-forward with residual
        ffn_output = self.ffn(x)
        x = self.norm2(x + ffn_output)
        
        return x

# Test
block = TransformerEncoderBlock(embed_dim=256, num_heads=8, d_ff=1024)
x = torch.randn(2, 10, 256)
out = block(x)
print(f"Input: {x.shape} -> Output: {out.shape}")

In [ ]:
class TransformerEncoder(nn.Module):
    """Stack of Transformer Encoder Blocks."""
    
    def __init__(self, num_layers, embed_dim, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        
    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

# BERT-base has 12 layers
encoder = TransformerEncoder(num_layers=6, embed_dim=768, num_heads=12, d_ff=3072)
x = torch.randn(2, 128, 768)
out = encoder(x)
print(f"6-layer encoder: {x.shape} -> {out.shape}")
print(f"Parameters: {sum(p.numel() for p in encoder.parameters()):,})")

## Exercises

### Exercise 1: Custom Implementation
Implement your own TransformerEncoderBlock WITHOUT using nn.MultiheadAttention.

## Summary
- ✅ Multi-head attention + residual + LayerNorm
- ✅ Position-wise feed-forward network
- ✅ Stack multiple blocks for deep encoder

## Next: [Task 5: Positional Embeddings](../task-05-positional-embeddings/overview.html)